# Challenge Sprint 4 — Classificação de Energia Renovável

**Integrantes (nome completo e RM):**

- NOME COMPLETO 1 — RM 00000
- NOME COMPLETO 2 — RM 00000
- NOME COMPLETO 3 — RM 00000

**Dataset:** Renewable Energy Production Dataset (2010–2020) — Kaggle
**Variável-alvo:** `Energy_Class`

Para rodar no Google Colab, envie o arquivo `Renewable_Energy_Data.csv` para `/content/` (painel de arquivos à esquerda).

Estrutura do notebook: (1) análise do dataset e correlação, (2) modelo linear e dois cenários de treino/teste, (3) avaliação de performance.

In [ ]:
import glob, os, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg") if os.environ.get("NO_DISPLAY") else None
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
                             precision_score, recall_score, classification_report)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42                 # mesma semente nos dois cenários
TARGET = "Energy_Class"
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
sns.set_theme(style="whitegrid")

## 1) Análise do dataset

### 1a) Carregamento do dataset em um DataFrame

In [ ]:
DATA_PATH = "/content/Renewable_Energy_Data.csv"   # caminho no Google Colab
df = pd.read_csv(DATA_PATH)
print("Arquivo:", DATA_PATH)
print("Formato:", df.shape)
display(df.head()) if "display" in globals() else print(df.head())
print()
df.info()
print("\nValores nulos por coluna:\n", df.isna().sum())
assert TARGET in df.columns, f"Coluna {TARGET} não encontrada. Colunas: {list(df.columns)}"
print("\nDistribuição da variável-alvo:\n", df[TARGET].value_counts())

### 1b) Encoding da variável-alvo (`Energy_Class`)

Os modelos do scikit-learn trabalham com valores numéricos. Usamos o `LabelEncoder`, que atribui um inteiro a cada classe (ordem alfabética das classes).

In [ ]:
le = LabelEncoder()
df["Energy_Class_enc"] = le.fit_transform(df[TARGET].astype(str))
mapping = {cls: int(i) for i, cls in enumerate(le.classes_)}
print("Mapeamento das classes:", mapping)

### 1c) Matriz de correlação (features + `Energy_Class` codificada)

Features numéricas são usadas diretamente. Se houver colunas categóricas (texto) de baixa cardinalidade (≤ 10 valores), aplicamos one-hot encoding; colunas de texto com muitos valores únicos (identificadores) são descartadas.

In [ ]:
raw_features = df.drop(columns=[TARGET, "Energy_Class_enc"])
num_cols = raw_features.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in raw_features.columns if c not in num_cols]

keep_cat = [c for c in cat_cols if raw_features[c].nunique() <= 10]
drop_cat = [c for c in cat_cols if c not in keep_cat]
if drop_cat:
    print("Colunas de texto descartadas (alta cardinalidade):", drop_cat)
if keep_cat:
    print("Colunas categóricas com one-hot:", keep_cat)

X_all = raw_features[num_cols].copy()
if keep_cat:
    X_all = pd.concat([X_all, pd.get_dummies(raw_features[keep_cat], drop_first=True).astype(int)], axis=1)
y = df["Energy_Class_enc"]
FEATURES_ALL = X_all.columns.tolist()
print("Features disponíveis:", FEATURES_ALL)

corr_df = pd.concat([X_all, y.rename("Energy_Class")], axis=1)
corr = corr_df.corr()

fig, ax = plt.subplots(figsize=(max(7, 0.9 * len(corr)), max(6, 0.8 * len(corr))))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, ax=ax)
ax.set_title("Matriz de correlação (Pearson) — features + Energy_Class (encoded)")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/matriz_correlacao.png", dpi=150)
plt.show()
corr.to_csv(f"{RESULTS_DIR}/matriz_correlacao.csv")

### 1d) Seleção de features: subconjunto ou todas?

Critérios técnicos usados para decidir:

1. **Relevância:** correlação (em módulo) de cada feature com `Energy_Class`. Correlações próximas de zero indicam pouca relação *linear* com o alvo.
2. **Redundância/multicolinearidade:** pares de features com |correlação| alta (> 0,90) carregam praticamente a mesma informação; manter ambas aumenta a instabilidade dos coeficientes de um modelo linear sem ganho preditivo.
3. **Evidência empírica:** comparar a acurácia por validação cruzada com todas as features vs. com o subconjunto.

Limitação importante: a correlação de Pearson mede apenas relação linear (e a classe codificada por `LabelEncoder` impõe uma ordem numérica que só faz sentido se as classes forem ordinais). Por isso a decisão é confirmada com a validação cruzada.

In [ ]:
REL_THRESHOLD = 0.10    # |corr| mínima com o alvo para considerar a feature relevante
REDUNDANCY_THRESHOLD = 0.90

corr_target = corr["Energy_Class"].drop("Energy_Class")
rank = corr_target.abs().sort_values(ascending=False)
print("Correlação de cada feature com Energy_Class (ordenada por |corr|):")
print(corr_target.reindex(rank.index).round(3).to_string())

# Pares redundantes
pairs = []
for i, a in enumerate(FEATURES_ALL):
    for b in FEATURES_ALL[i+1:]:
        if abs(corr.loc[a, b]) > REDUNDANCY_THRESHOLD:
            pairs.append((a, b, corr.loc[a, b]))
print("\nPares com |corr| >", REDUNDANCY_THRESHOLD, ":", [(a, b, round(c, 3)) for a, b, c in pairs] or "nenhum")

# Subconjunto candidato: relevantes e sem redundância (mantém a mais correlacionada com o alvo)
selected = [f for f in FEATURES_ALL if abs(corr_target[f]) >= REL_THRESHOLD]
for a, b, _ in pairs:
    if a in selected and b in selected:
        selected.remove(a if abs(corr_target[a]) < abs(corr_target[b]) else b)
if not selected:                       # segurança: nunca ficar sem features
    selected = FEATURES_ALL.copy()

pipe_cv = lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
cv_all = cross_val_score(pipe_cv(), X_all[FEATURES_ALL], y, cv=5, scoring="accuracy")
cv_sel = cross_val_score(pipe_cv(), X_all[selected], y, cv=5, scoring="accuracy")
print(f"\nAcurácia CV (5-fold) com TODAS ({len(FEATURES_ALL)}): {cv_all.mean():.4f} ± {cv_all.std():.4f}")
print(f"Acurácia CV (5-fold) com SUBCONJUNTO ({len(selected)}): {cv_sel.mean():.4f} ± {cv_sel.std():.4f}")

if len(selected) == len(FEATURES_ALL):
    decision = "USAR TODAS AS FEATURES"
    FEATURES = FEATURES_ALL
elif cv_sel.mean() >= cv_all.mean() - 0.01:
    decision = "USAR SUBCONJUNTO"
    FEATURES = selected
else:
    decision = "USAR TODAS AS FEATURES"
    FEATURES = FEATURES_ALL
print("\nDECISÃO:", decision)
print("Features utilizadas no modelo:", FEATURES)
pd.DataFrame({"feature": FEATURES_ALL, "corr_com_alvo": corr_target[FEATURES_ALL].values,
              "usada_no_modelo": [f in FEATURES for f in FEATURES_ALL]}
             ).to_csv(f"{RESULTS_DIR}/selecao_features.csv", index=False)

**Justificativa técnica (resultados obtidos neste dataset):**

- **Correlação com o alvo é fraca para todas as features.** A maior é `Efficiency_Ratio` (−0,167); as demais ficam abaixo de 0,05 em módulo. Nenhuma feature isolada explica bem a classe. Com o limiar de 0,10, apenas `Efficiency_Ratio` seria mantida.
- **Não há redundância relevante:** nenhum par de features tem |corr| > 0,90 (o maior valor entre as features numéricas é 0,26, entre `Temperature_C` e `Efficiency_Ratio`; entre dummies, o máximo é 0,49). Logo, não há features para eliminar por multicolinearidade.
- **A validação cruzada confirma:** com todas as 15 features (7 numéricas + dummies de Region, Energy_Source e Season) a acurácia média (5-fold) é ≈ 0,779; só com `Efficiency_Ratio`, cai para ≈ 0,762 e fica bem mais instável (desvio 0,038 vs 0,012).
- **Decisão: usar todas as features.** A baixa correlação linear com o alvo não significa que as features sejam inúteis: o modelo combina várias features ao mesmo tempo (relações multivariadas), e a correlação de Pearson com um alvo codificado por `LabelEncoder` (ordem alfabética: High=0, Low=1, Medium=2) não representa uma ordem real entre as classes. Como não há colinearidade e o conjunto completo é mais acurado e mais estável, descartar features só traria perda de informação.

## 2) Definição e aplicação do modelo

### 2a) Modelo escolhido: **Regressão Logística** (`sklearn.linear_model.LogisticRegression`)

### 2b) Por que é um modelo linear?

A Regressão Logística calcula uma combinação linear das features, $z = w_1x_1 + w_2x_2 + \dots + w_nx_n + b$, e passa o resultado por uma função sigmoide (ou softmax, no caso multiclasse) para obter probabilidades. A classe é decidida comparando probabilidades; no caso binário, o limiar de 0,5 corresponde a $z = 0$, ou seja, a fronteira de decisão é $w \cdot x + b = 0$ — um **hiperplano** (uma reta em 2D, um plano em 3D). A sigmoide é não linear, mas é monotônica e só transforma a saída; **a fronteira que separa as classes continua sendo linear no espaço das features**. No caso multiclasse, há um vetor de pesos por classe e as regiões de decisão são delimitadas por hiperplanos (as fronteiras entre cada par de classes também são lineares). Por isso o modelo só separa bem classes que sejam aproximadamente linearmente separáveis.

### 2c/2d) Treino com as features selecionadas em dois cenários

- **Cenário 1:** 60% treino / 40% teste
- **Cenário 2:** 85% treino / 15% teste

Para comparação justa, tudo é mantido igual (mesmo modelo, hiperparâmetros, pré-processamento, `random_state` e divisão estratificada); só muda `test_size`. O pré-processamento (imputação + padronização) está dentro do `Pipeline`, ajustado somente no treino, evitando vazamento de dados.

In [ ]:
def run_scenario(test_size, name):
    X = X_all[FEATURES]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=RANDOM_STATE, stratify=y)
    model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                          LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    return dict(name=name, test_size=test_size, n_train=len(X_tr), n_test=len(X_te),
                y_te=y_te, pred=pred, model=model)

sc1 = run_scenario(0.40, "Cenário 1 (60/40)")
sc2 = run_scenario(0.15, "Cenário 2 (85/15)")
for s in (sc1, sc2):
    print(f"{s['name']}: treino={s['n_train']} amostras | teste={s['n_test']} amostras")

## 3) Avaliação da performance

### 3a) Matriz de confusão de cada cenário

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, s in zip(axes, (sc1, sc2)):
    cm = confusion_matrix(s["y_te"], s["pred"])
    ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{s['name']} — teste={s['n_test']}")
    pd.DataFrame(cm, index=[f"real_{c}" for c in le.classes_],
                 columns=[f"pred_{c}" for c in le.classes_]).to_csv(
        f"{RESULTS_DIR}/matriz_confusao_{int(s['test_size']*100)}pct_teste.csv")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/matrizes_confusao.png", dpi=150)
plt.show()

### 3b) Acurácia, precisão e recall por cenário

In [ ]:
# Para problema multiclasse, precisão e recall são médias entre as classes.
# macro: média simples (trata todas as classes igualmente) | weighted: ponderada pelo suporte.
rows = []
for s in (sc1, sc2):
    rows.append({
        "Cenário": s["name"], "Teste (%)": int(s["test_size"] * 100),
        "N treino": s["n_train"], "N teste": s["n_test"],
        "Acurácia": accuracy_score(s["y_te"], s["pred"]),
        "Precisão (macro)": precision_score(s["y_te"], s["pred"], average="macro", zero_division=0),
        "Recall (macro)": recall_score(s["y_te"], s["pred"], average="macro", zero_division=0),
        "Precisão (weighted)": precision_score(s["y_te"], s["pred"], average="weighted", zero_division=0),
        "Recall (weighted)": recall_score(s["y_te"], s["pred"], average="weighted", zero_division=0),
    })
    with open(f"{RESULTS_DIR}/classification_report_{int(s['test_size']*100)}pct_teste.txt", "w") as f:
        f.write(classification_report(s["y_te"], s["pred"], target_names=le.classes_, zero_division=0))
    print(s["name"]); print(classification_report(s["y_te"], s["pred"], target_names=le.classes_, zero_division=0))

metrics = pd.DataFrame(rows)
metrics.to_csv(f"{RESULTS_DIR}/metricas_cenarios.csv", index=False)
metrics.round(4)

In [ ]:
ax = metrics.set_index("Cenário")[["Acurácia", "Precisão (macro)", "Recall (macro)"]].plot(
    kind="bar", figsize=(8, 5), rot=0)
ax.set_ylim(0, 1.05); ax.set_title("Métricas por cenário"); ax.set_ylabel("Valor")
for c in ax.containers: ax.bar_label(c, fmt="%.3f", fontsize=8)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/comparacao_metricas.png", dpi=150); plt.show()

### 3c) O desempenho é adequado?

Baseline (sempre prever a classe majoritária, `High`) = 48,6% de acurácia. O modelo supera esse baseline nos dois cenários (+22,9 p.p. no Cenário 1 e +30,1 p.p. no Cenário 2), então ele aprende algo real das features.

Resultados: Cenário 1 (60/40) — acurácia 0,715, precisão macro 0,684, recall macro 0,637. Cenário 2 (85/15) — acurácia 0,787, precisão macro 0,794, recall macro 0,748.

**É adequado?** Parcialmente. Para um primeiro modelo linear simples, ~71–79% é razoável e bem acima do acaso, mas as métricas por classe mostram fraquezas: a classe `High` é muito bem reconhecida (recall 0,94 e 0,92), enquanto `Low` e `Medium` sofrem. No Cenário 1, o recall de `Low` é 0,44 e o de `Medium`, 0,53: 28 das 71 amostras `Low` foram classificadas como `Medium`, e 46 das 134 `Medium` foram classificadas como `High`. O modelo tende a "empurrar" os casos para a classe majoritária. Se errar as classes de menor produção tiver custo operacional relevante, esse desempenho não seria suficiente, e seria recomendável testar modelos não lineares ou balanceamento de classes.

In [ ]:
baseline = y.value_counts(normalize=True).max()
print(f"Baseline (sempre prever a classe majoritária): {baseline:.4f}")
for _, r in metrics.iterrows():
    gain = r["Acurácia"] - baseline
    print(f"{r['Cenário']}: acurácia={r['Acurácia']:.4f} | ganho sobre baseline={gain:+.4f} | "
          f"precisão macro={r['Precisão (macro)']:.4f} | recall macro={r['Recall (macro)']:.4f}")
a1, a2 = metrics["Acurácia"]
print(f"\nDiferença de acurácia (15% teste − 40% teste): {a2 - a1:+.4f}")

### 3d) Comparação entre 40% e 15% de teste

| Métrica | 40% teste (n=400) | 15% teste (n=150) |
|---|---|---|
| Acurácia | 0,715 | 0,787 |
| Precisão (macro) | 0,684 | 0,794 |
| Recall (macro) | 0,637 | 0,748 |

O Cenário 2 (85/15) apresentou métricas melhores em todas as medidas (+7,2 p.p. de acurácia). Duas causas possíveis, que se somam: (i) o modelo foi treinado com mais dados (850 vs 600 amostras); (ii) o teste de 150 amostras é pequeno e as divisões têm amostras diferentes, então parte da diferença pode ser apenas variação amostral. Os intervalos de confiança (95%) mostram isso: ≈ [0,671; 0,759] no Cenário 1 e ≈ [0,721; 0,852] no Cenário 2 — eles se sobrepõem parcialmente, ou seja, **não dá para afirmar com segurança que o modelo do Cenário 2 é realmente melhor**; a estimativa do Cenário 1 é mais confiável. Além disso, a validação cruzada com todas as features deu ≈ 0,779, valor mais próximo do Cenário 2, mas a média de 5 folds é uma estimativa mais robusta do que qualquer uma das duas divisões isoladas.

### 3e) Vantagens e limitações de um conjunto de teste maior ou menor

**Teste maior (40%)**
- *Vantagens:* estimativa de desempenho mais confiável e estável (menor variância; intervalo de confiança mais estreito); classes minoritárias aparecem em quantidade suficiente para métricas por classe significativas.
- *Limitações:* sobram menos dados para treino (60%), o que pode gerar um modelo pior que o possível (estimativa *pessimista* do desempenho do modelo final); em datasets pequenos, o treino pode não cobrir bem a variabilidade dos dados.

**Teste menor (15%)**
- *Vantagens:* mais dados para treino (85%), o que tende a produzir um modelo melhor, com mais exemplos de cada classe e fronteira mais bem estimada.
- *Limitações:* poucas amostras de teste tornam as métricas ruidosas — uma única amostra a mais ou a menos altera sensivelmente acurácia/recall, e classes raras podem ter pouquíssimos exemplos (ou nenhum) no teste. A estimativa fica menos confiável e mais sujeita à "sorte" da divisão.

**Conclusão:** há um trade-off viés–variância na avaliação: mais teste → estimativa mais confiável, modelo treinado com menos dados; menos teste → melhor modelo, avaliação mais incerta. Uma alternativa mais robusta é a validação cruzada (k-fold), que usa todos os dados para treino e teste em rodadas diferentes.

In [ ]:
# Evidência extra para 3e: incerteza da acurácia em função do tamanho do teste (erro padrão binomial)
for s, (_, r) in zip((sc1, sc2), metrics.iterrows()):
    p, n = r["Acurácia"], r["N teste"]
    se = (p * (1 - p) / n) ** 0.5
    print(f"{s['name']}: acurácia={p:.3f} | n_teste={n} | erro padrão ≈ {se:.3f} | IC95% ≈ [{p-1.96*se:.3f}, {p+1.96*se:.3f}]")